# Training Plan Multi-Agent Notebook

Simple local-only prototype for a three-step training-plan workflow:

1. **Explain required inputs** so the user knows what to provide.
2. **Gather requirements** from a user brief into a structured profile.
3. **Generate a plan** from those requirements.

Storage is a local JSON file for now. There are no MCP tools, file-search tools, hosted connectors, or external data sources. The notebook loads `OPENAI_API_KEY` from `.env` via `python-dotenv` and uses a small, cheap model by default.

In [2]:
from __future__ import annotations

import json
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

from agents import Agent, ModelSettings, Runner
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to .env or your shell environment.")

# Override in .env with TRAINING_PLAN_MODEL=... if you want a different cheap model.
MODEL = os.getenv("TRAINING_PLAN_MODEL", "gpt-4o-mini")
MEMORY_FILE = Path.cwd() / "training_plan_memory.json"

MODEL


'gpt-4o-mini'

## Structured Output Schemas

All agents return Pydantic models. The JSON memory stores these model dumps plus local record IDs.

In [3]:
class IntakeGuide(BaseModel):
    opening_message: str = Field(description="Brief, friendly explanation of what the user should provide.")
    required_fields: list[str] = Field(description="Information needed before requirements can be gathered well.")
    optional_fields: list[str] = Field(default_factory=list, description="Useful but non-blocking information.")
    example_brief: str = Field(description="A concise example user brief the user can copy and edit.")
    why_this_matters: list[str] = Field(default_factory=list, description="Short reasons these inputs improve the plan.")


class GoalRace(BaseModel):
    race_name: str | None = Field(default=None, description="Race name if known.")
    race_distance: str = Field(description="Race distance, e.g. 5K, 10K, half marathon, marathon.")
    race_date: str = Field(description="Goal race date in ISO format if known; otherwise the user's phrase.")
    goal_time: str | None = Field(default=None, description="Goal finish time, if supplied.")
    priority: Literal["A", "B", "C", "unknown"] = "unknown"


class TrainingConstraints(BaseModel):
    runs_per_week: int | None = Field(default=None, ge=1, le=7)
    preferred_run_days: list[str] = Field(default_factory=list)
    non_negotiables: list[str] = Field(default_factory=list)
    unavailable_days: list[str] = Field(default_factory=list)
    cross_training: list[str] = Field(default_factory=list)
    max_long_run_day: str | None = None


class FitnessBackground(BaseModel):
    current_fitness_level: Literal[
        "beginner",
        "returning",
        "recreational",
        "intermediate",
        "advanced",
        "unknown",
    ] = "unknown"
    current_weekly_distance_km: float | None = Field(default=None, ge=0)
    recent_long_run_km: float | None = Field(default=None, ge=0)
    recent_race_results: list[str] = Field(default_factory=list)
    injury_history_or_limitations: list[str] = Field(default_factory=list)
    easy_pace_or_hr: str | None = None
    training_age: str | None = Field(default=None, description="How long the user has been running consistently.")


class TrainingRequirements(BaseModel):
    athlete_name: str | None = None
    goal_race: GoalRace | None = None
    constraints: TrainingConstraints
    background: FitnessBackground
    available_plan_weeks: int | None = Field(default=None, ge=1, le=80)
    missing_information: list[str] = Field(default_factory=list)
    clarifying_questions: list[str] = Field(default_factory=list)
    assumptions: list[str] = Field(default_factory=list)
    ready_for_plan: bool
    notes_for_planner: str = ""


class TrainingSession(BaseModel):
    day: str
    session_type: Literal[
        "rest",
        "easy",
        "steady",
        "tempo",
        "intervals",
        "long_run",
        "race",
        "strength",
        "mobility",
        "cross_training",
    ]
    distance_km: float | None = Field(default=None, ge=0)
    duration_minutes: int | None = Field(default=None, ge=0)
    intensity: str
    purpose: str
    details: str


class TrainingWeek(BaseModel):
    week_number: int = Field(ge=1)
    focus: str
    estimated_distance_km: float | None = Field(default=None, ge=0)
    sessions: list[TrainingSession]
    coach_notes: list[str] = Field(default_factory=list)


class TrainingPlan(BaseModel):
    plan_id: str
    requirements_id: str
    created_at: str
    athlete_name: str | None = None
    goal_summary: str
    plan_weeks: int = Field(ge=1, le=80)
    assumptions: list[str] = Field(default_factory=list)
    weeks: list[TrainingWeek]
    risk_flags: list[str] = Field(default_factory=list)
    next_review_prompt: str


## Local JSON Memory

This is intentionally simple for now: append-only `intake_guides`, `requirements`, and `plans` arrays in `training_plan_memory.json`.

In [4]:
def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def new_id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:12]}"


def empty_memory() -> dict:
    return {"intake_guides": [], "requirements": [], "plans": []}


def load_memory() -> dict:
    if not MEMORY_FILE.exists():
        return empty_memory()

    with MEMORY_FILE.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, dict):
        return empty_memory()
    data.setdefault("intake_guides", [])
    data.setdefault("requirements", [])
    data.setdefault("plans", [])
    return data


def save_memory(memory: dict) -> None:
    tmp_file = MEMORY_FILE.with_suffix(".tmp")
    with tmp_file.open("w", encoding="utf-8") as f:
        json.dump(memory, f, indent=2, ensure_ascii=False)
        f.write("\n")
    tmp_file.replace(MEMORY_FILE)


def remember_intake_guide(intake_guide: IntakeGuide, user_context: str) -> str:
    memory = load_memory()
    intake_guide_id = new_id("intake")
    memory["intake_guides"].append(
        {
            "id": intake_guide_id,
            "created_at": utc_now(),
            "user_context": user_context,
            "data": intake_guide.model_dump(mode="json"),
        }
    )
    save_memory(memory)
    return intake_guide_id


def remember_requirements(requirements: TrainingRequirements, source_text: str) -> str:
    memory = load_memory()
    requirements_id = new_id("req")
    memory["requirements"].append(
        {
            "id": requirements_id,
            "created_at": utc_now(),
            "source_text": source_text,
            "data": requirements.model_dump(mode="json"),
        }
    )
    save_memory(memory)
    return requirements_id


def remember_plan(plan: TrainingPlan) -> None:
    memory = load_memory()
    memory["plans"].append(
        {
            "id": plan.plan_id,
            "created_at": plan.created_at,
            "requirements_id": plan.requirements_id,
            "data": plan.model_dump(mode="json"),
        }
    )
    save_memory(memory)


load_memory()


{'intake_guides': [], 'requirements': [], 'plans': []}

## Agents

There are three agents and no tools. The first explains what input is needed, the second turns a free-form user brief into requirements, and the third turns complete requirements into a plan.

In [5]:
intake_agent = Agent(
    name="Training Intake Guide Agent",
    model=MODEL,
    model_settings=ModelSettings(temperature=0.1),
    output_type=IntakeGuide,
    instructions=(
        "You help a runner understand what information to provide before a training plan can be built. "
        "Return a concise checklist, useful optional details, and a copy-editable example brief. "
        "Ask for concrete values where possible: runs per week, non-negotiables, current fitness, "
        "current weekly distance, recent long run, goal race date, race distance, goal time, injuries, "
        "unavailable days, preferred long-run day, and strength or cross-training preferences. "
        "Do not gather requirements yet. Do not call tools, browse, or use connectors."
    ),
)


requirements_agent = Agent(
    name="Training Requirements Agent",
    model=MODEL,
    model_settings=ModelSettings(temperature=0.1),
    output_type=TrainingRequirements,
    instructions=(
        "You gather requirements for a running training plan. "
        "Use only the user's supplied text. Do not call tools, browse, or assume external data. "
        "Extract runs per week, non-negotiables, current fitness level, race date, goal time, "
        "injury constraints, current mileage, recent long run, and schedule constraints. "
        "If critical information is missing, set ready_for_plan=false and ask concise clarifying questions. "
        "If enough information is present, set ready_for_plan=true and list any mild assumptions explicitly."
    ),
)


planner_agent = Agent(
    name="Training Plan Agent",
    model=MODEL,
    model_settings=ModelSettings(temperature=0.2),
    output_type=TrainingPlan,
    instructions=(
        "You generate conservative running training plans from structured requirements. "
        "Use only the provided requirements JSON. Do not call tools, browse, or use connectors. "
        "Respect non-negotiables and requested runs per week. Build a practical week-by-week plan with rest, "
        "easy running, quality work, long runs, and tapering where appropriate. "
        "Avoid aggressive mileage jumps and flag risks when goals appear ambitious or information is incomplete."
    ),
)


## Orchestration

No looping yet: `start_training_plan_intake()` runs the initial guidance step and stops for human input. `build_training_plan()` then runs the requirements step once, stores it, and only calls the planner if `ready_for_plan` is true.

In [6]:
async def start_training_plan_intake(user_context: str = "") -> tuple[str, IntakeGuide]:
    prompt = (
        "Explain what information the user should provide before you gather training requirements. "
        "If user context is supplied, tailor the checklist lightly to that context.\n\n"
        f"User context: {user_context or 'None'}"
    )
    result = await Runner.run(intake_agent, input=prompt)
    intake_guide = result.final_output_as(IntakeGuide)
    intake_guide_id = remember_intake_guide(intake_guide, user_context)
    return intake_guide_id, intake_guide


async def gather_requirements(user_brief: str) -> tuple[str, TrainingRequirements]:
    result = await Runner.run(requirements_agent, input=user_brief)
    requirements = result.final_output_as(TrainingRequirements)
    requirements_id = remember_requirements(requirements, user_brief)
    return requirements_id, requirements


async def generate_plan(requirements_id: str, requirements: TrainingRequirements) -> TrainingPlan:
    if not requirements.ready_for_plan:
        raise ValueError("Requirements are not ready for planning yet.")

    plan_id = new_id("plan")
    planner_input = {
        "plan_id": plan_id,
        "requirements_id": requirements_id,
        "created_at": utc_now(),
        "requirements": requirements.model_dump(mode="json"),
    }

    result = await Runner.run(
        planner_agent,
        input=json.dumps(planner_input, indent=2),
    )
    plan = result.final_output_as(TrainingPlan)
    remember_plan(plan)
    return plan


async def build_training_plan(user_brief: str) -> dict:
    requirements_id, requirements = await gather_requirements(user_brief)

    if not requirements.ready_for_plan:
        return {
            "status": "needs_more_information",
            "requirements_id": requirements_id,
            "requirements": requirements,
            "clarifying_questions": requirements.clarifying_questions,
        }

    plan = await generate_plan(requirements_id, requirements)
    return {
        "status": "plan_generated",
        "requirements_id": requirements_id,
        "requirements": requirements,
        "plan": plan,
    }


## Start Here: Intake Guide

Run this first. It tells the user what to provide, stores the guide locally, and then stops for human input.

In [7]:
intake_guide_id, intake_guide = await start_training_plan_intake()

print(intake_guide.opening_message)
print("\nRequired:")
for field in intake_guide.required_fields:
    print(f"- {field}")

if intake_guide.optional_fields:
    print("\nOptional but useful:")
    for field in intake_guide.optional_fields:
        print(f"- {field}")

print("\nExample brief:")
print(intake_guide.example_brief)
print(f"\nStored intake guide: {intake_guide_id}")


To create a personalized training plan, please provide the following information about your running goals and current fitness level.

Required:
- Runs per week
- Current fitness level (beginner, intermediate, advanced)
- Current weekly distance (miles or kilometers)
- Recent long run distance
- Goal race date
- Race distance
- Goal time
- Injuries (if any)
- Unavailable days for training
- Preferred long-run day

Optional but useful:
- Strength training preferences
- Cross-training activities
- Preferred running routes
- Nutrition considerations

Example brief:
I run 4 times a week and currently cover about 20 miles. My recent long run was 8 miles. I'm training for a half marathon on June 15, aiming for a time of 1:45. I have no injuries but can't train on Sundays. I prefer to do my long runs on Saturdays. I also want to include some strength training in my routine.

Stored intake guide: intake_c129b4f5d386


## Example Run

After the user answers the intake guide, paste their details into `USER_BRIEF`, then run the cell. If the brief is incomplete, the notebook stores the structured requirements and returns clarifying questions instead of generating a plan.

In [8]:
USER_BRIEF = """
My name is Andrew. I am training for the Cape Town Marathon on 2026-10-18.
I want to run 3:30. I can run 5 days per week. Sunday long runs are non-negotiable,
and I cannot run on Fridays. I am currently running about 45 km per week with a 20 km
long run. I have been running consistently for 3 years. No current injuries, but I want
to keep strength training on Wednesdays.
""".strip()

result = await build_training_plan(USER_BRIEF)
result["status"]


'plan_generated'

In [9]:
if result["status"] == "needs_more_information":
    print("Questions to ask next:")
    for question in result["clarifying_questions"]:
        print(f"- {question}")
else:
    plan = result["plan"]
    print(plan.goal_summary)
    print(f"Stored plan: {plan.plan_id}")
    print(f"Weeks: {plan.plan_weeks}")
    print(f"Memory file: {MEMORY_FILE}")


Prepare for Cape Town Marathon aiming for a 3:30 finish.
Stored plan: plan_07187d7b84de
Weeks: 22
Memory file: /Users/andrewallkin/personal/repos/strava-automation-service/training_plan_memory.json


In [10]:
# Inspect the latest saved records.
memory = load_memory()
{
    "intake_guides_count": len(memory["intake_guides"]),
    "requirements_count": len(memory["requirements"]),
    "plans_count": len(memory["plans"]),
    "latest_intake_guide_id": memory["intake_guides"][-1]["id"] if memory["intake_guides"] else None,
    "latest_requirement_id": memory["requirements"][-1]["id"] if memory["requirements"] else None,
    "latest_plan_id": memory["plans"][-1]["id"] if memory["plans"] else None,
}


{'intake_guides_count': 1,
 'requirements_count': 1,
 'plans_count': 1,
 'latest_intake_guide_id': 'intake_c129b4f5d386',
 'latest_requirement_id': 'req_b7c3943885e1',
 'latest_plan_id': 'plan_07187d7b84de'}